# Parsimonious models

Select features using development data only, refit reduced logistic-regression models, and evaluate them on the untouched temporal test set.

The original target column is used during fitting so `fit_model()` applies the configured ESBL/non-ESBL mapping. `unknown` outcomes are excluded.

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display

from icare_risk.data.audit import load_admission_file
from icare_risk.data.feature_sets import resolve_feature_set
from icare_risk.models.training import fit_model, encode_binary_target
from icare_risk.models.prediction import predict_probability
from icare_risk.explainability.feature_selection import rank_pipeline_features, feature_set_table
from icare_risk.evaluation.metrics import compute_metrics

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)
print(f"Using configuration: {CONFIG_PATH}")


Using configuration: ../src/icare_risk/config/icare/analysis.yaml


In [18]:
target = config["columns"]["target"]
target_mapping = config.get("target_mapping")

df = load_admission_file(config["outputs"]["processed_dataset"])
required = [target, "split"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

df["_target_numeric"] = encode_binary_target(
    df[target],
    positive_values=target_mapping.get("positive_values"),
    negative_values=target_mapping.get("negative_values"),
)

n_before = len(df)
df = df.loc[df["_target_numeric"].notna()].copy()
print(f"Excluded unresolved outcomes: {n_before - len(df):,}")

development = df.loc[df["split"].isin(["train", "validation"])].copy()
test = df.loc[df["split"].eq("test")].copy()
print(f"Development rows: {len(development):,}; test rows: {len(test):,}")
print(f"Development events: {int(development['_target_numeric'].sum()):,}")
print(f"Test events: {int(test['_target_numeric'].sum()):,}")


Excluded unresolved outcomes: 0
Development rows: 7,517; test rows: 2,483
Development events: 1,442
Test events: 508


In [19]:
feature_cfg = config["feature_sets"].get("all_available_baseline")
if feature_cfg is None:
    raise KeyError("analysis.yaml must define feature_sets.all_available_baseline")

features = resolve_feature_set(
    df,
    feature_cfg,
    exclude_columns=[target, "split", "_target_numeric"],
)
print(f"Resolved {len(features)} baseline features")
display(pd.DataFrame({"feature": features}))


Resolved 170 baseline features


,feature
0,AGE_AT_ADMISSION
1,MAIN_SPECIALTY_CODE
2,MAIN_SPECIALTY_CODE_DESC
3,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE
4,index_admission
...,...
165,days_since_last_discharge
166,ETHNICITY
167,ETHNICITY_DES
168,GENDER


In [20]:
ranking_cfg = config.get("explainability", {})
ranking_model = fit_model(
    development,
    features,
    target,
    model_type=ranking_cfg.get("ranking_model_type", "logistic_regression"),
    params=ranking_cfg.get("ranking_model_params", {"C": 1.0}),
    target_mapping=target_mapping,
)
ranking = rank_pipeline_features(ranking_model, features)
display(ranking.head(30))

sizes = tuple(ranking_cfg.get("parsimonious_feature_sizes", [5, 10, 20, 30]))
feature_sets = feature_set_table(ranking, sizes=sizes)
print("Feature sets:", {name: len(cols) for name, cols in feature_sets.items()})


/usr/local/lib/python3.10/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['kim_hx_urinary_catheter_dslast' 'kim_hx_urinary_catheter_dsfirst'
 'tumbarello_hx_urinary_catheter_dslast'
 'tumbarello_hx_urinary_catheter_dsfirst'
 'gavaghan_hx_urinary_catheter_dslast'
 'gavaghan_hx_urinary_catheter_dsfirst' 'days_since_last_discharge']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,rank,feature,importance
0,1,charlson_hx_leukemia_cnt,0.908954
1,2,charlson_hx_leukemia,0.908954
2,3,charlson_hx_pud_cnt,0.716312
3,4,charlson_hx_pud,0.716312
4,5,charlson_hx_liver_mod_sev_cnt,0.618119
5,6,charlson_hx_liver_mod_sev,0.618119
6,7,kim_score,0.499110
7,8,charlson_hx_diabetes_comp_cnt,0.356947
8,9,charlson_hx_diabetes_comp,0.356947
9,10,gavaghan_hx_prior_antibiotics_90d,0.270811


Feature sets: {'full': 156, 'top_5': 5, 'top_10': 10, 'top_20': 20, 'top_30': 30}


In [21]:
threshold = float(config.get("analysis", {}).get("evaluation_threshold", 0.5))
rows = []

for feature_set_name, selected_features in feature_sets.items():
    print(f"Fitting {feature_set_name} ({len(selected_features)} features)...")
    fitted = fit_model(
        development,
        selected_features,
        target,
        model_type="logistic_regression",
        params=ranking_cfg.get("ranking_model_params", {"C": 1.0}),
        target_mapping=target_mapping,
    )
    probability = predict_probability(fitted, test, selected_features)
    metrics = compute_metrics(
        test["_target_numeric"],
        probability,
        threshold=threshold,
    )
    rows.append({
        "model": f"lr_{feature_set_name}",
        "n_features": len(selected_features),
        **metrics,
    })

parsimonious_results = (
    pd.DataFrame(rows)
    .sort_values(["auroc", "auprc"], ascending=False, na_position="last")
    .reset_index(drop=True)
)
display(parsimonious_results)


Fitting full (156 features)...
Fitting top_5 (5 features)...
Fitting top_10 (10 features)...
Fitting top_20 (20 features)...
Fitting top_30 (30 features)...


,model,n_features,n,n_events,event_rate,auroc,auprc,brier_score,threshold,tp,tn,fp,fn,sensitivity,specificity,ppv,npv,accuracy,balanced_accuracy,f1
0,lr_full,156,0,0,NaN,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,lr_top_5,5,0,0,NaN,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lr_top_10,10,0,0,NaN,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,lr_top_20,20,0,0,NaN,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,lr_top_30,30,0,0,NaN,NaN,NaN,NaN,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
output_dir = Path(
    config["outputs"].get("feature_importance_directory", "results/feature_importance")
)
output_dir.mkdir(parents=True, exist_ok=True)

ranking.to_csv(output_dir / "development_feature_ranking.csv", index=False)
parsimonious_results.to_csv(
    output_dir / "parsimonious_model_results.csv",
    index=False,
)

for name, selected in feature_sets.items():
    pd.DataFrame({"feature": selected}).to_csv(
        output_dir / f"{name}_features.csv",
        index=False,
    )

print(f"Saved results to: {output_dir.resolve()}")


Saved results to: /app/notebooks/results/feature_importance
